# Lab 14A: Reading PyTorch code – shapes and parameters

Many exam questions show a short PyTorch program and ask what it prints: a shape, a number of parameters,
or a value. This notebook trains exactly that skill.

**How to work:** each exercise asks a question about a short code cell. Think about the answer first, then run the
cell and compare. The explanation is in the collapsible block below each cell.

The PyTorch and Python you need is listed in the *PyTorch note for the exam*. Runs on a CPU in a few minutes;
no data is downloaded.

| Part | Topic | Learning goals |
|---|---|---|
| 0 | Tensors and the batch dimension | read programs (P1) |
| 1 | Layers and output shapes | read programs, sizes (P1, P3) |
| 2 | Counting parameters | sizes and costs (P3) |
| 3 | Beyond CNNs: 1×1 convolutions, RNNs, ViT tokens, attention | read programs, by hand (P1, P4) |

Each part starts with the formulas you need.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
_ = torch.manual_seed(0)

def count(module):
    # number of learnable values in a module: numel() = number of elements of each parameter tensor
    return sum(p.numel() for p in module.parameters())

---
## Part 0 – Tensors and the batch dimension

Images are stored as **(N, C, H, W)**: batch, channels, height, width. The first dimension is *always* the batch, also when it is 1.

### Exercise 0.1 – Indexing and flattening

**What are the three printed shapes?**

In [ ]:
x = torch.randn(8, 3, 64, 48)        # a batch of 8 RGB images, 64×48 pixels
print(x.shape)
print(x[0].shape)                    # the first image of the batch
print(x.flatten(1).shape)            # flatten everything except dimension 0

<details><summary><b>Show the answer</b></summary>

`x[0]` removes the batch dimension: one image, `(3, 64, 48)`. `flatten(1)` keeps dimension 0 (the batch) and
multiplies the rest: 3·64·48 = 9,216. This is what `nn.Flatten()` does before a `Linear` layer.

</details>

### Exercise 0.2 – Concatenating and transposing

Sequences are stored as **(N, T, D)**: batch, time steps (tokens), features.

**What do the two lines print?**

In [ ]:
a = torch.randn(2, 5, 16)      # 2 sequences, 5 tokens, 16 features
b = torch.randn(2, 3, 16)      # 2 sequences, 3 tokens, 16 features
print(torch.cat([a, b], dim=1).shape)
print(a.transpose(1, 2).shape)

<details><summary><b>Show the answer</b></summary>

`torch.cat` along dim 1 joins the sequences: 5 + 3 = 8 tokens. All other dimensions must match. `transpose(1, 2)`
swaps dimensions 1 and 2. In a U-Net, the skip connection uses `torch.cat(..., dim=1)` on `(N, C, H, W)` tensors, i.e. it
stacks *channels*.

</details>

### Exercise 0.3 – Softmax along a dimension

**Two images, three classes. Which of the two printed tensors contains only ones?**

In [ ]:
z = torch.tensor([[1.0, 2.0, 3.0],
                  [0.0, 0.0, 0.0]])          # class scores: 2 images, 3 classes
p = torch.softmax(z, dim=1)
print(p.sum(dim=1))
print(p.sum(dim=0))

<details><summary><b>Show the answer</b></summary>

`dim=1` normalises each **row** (each image) over its classes, so every row sums to 1: the first line is `tensor([1., 1.])`.
The column sums (second line) have no meaning. With `dim=0` the softmax would normalise over the *images* – a typical bug.

</details>

---
## Part 1 – Layers and output shapes

**Formulas.** For an input of height $H$ (the same for the width $W$), kernel size $K$, stride $S$ and padding $P$:

| Layer | Output height | Output channels |
|---|---|---|
| `Conv2d(C_in, C_out, K, S, P)` | $H' = \left\lfloor \dfrac{H + 2P - K}{S} \right\rfloor + 1$ | $C_{out}$ |
| `MaxPool2d(K, S, P)` / `AvgPool2d` | same formula (default $S = K$, $P = 0$) | unchanged |
| `ConvTranspose2d(C_in, C_out, K, S, P)` | $H' = (H - 1)\,S - 2P + K$ | $C_{out}$ |
| `AdaptiveAvgPool2d(1)` (global average pooling) | $1$ | unchanged |
| `Flatten()` on $(N, C, H, W)$ | gives $(N,\; C \cdot H \cdot W)$ | – |
| `Linear(D_in, D_out)` on $(N, D_{in})$ | gives $(N, D_{out})$ | – |

- ⌊ ⌋ means rounding **down**. The batch size $N$ never changes.
- “Same” padding for stride 1 (output size = input size): $P = (K - 1)/2$, e.g. $K = 3 \Rightarrow P = 1$, $K = 5 \Rightarrow P = 2$.
- `MaxPool2d(kernel_size=2)` has stride 2 and halves $H$ and $W$.
- Activation functions (`ReLU`, …), `BatchNorm` and `Dropout` do not change the shape.

### Exercise 1.1 – Tracing a small CNN

**What is the shape after every layer?**

In [ ]:
net = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=12, kernel_size=5, stride=1, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(in_channels=12, out_channels=24, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Flatten(),
)
x = torch.randn(4, 3, 40, 40)      # (batch, channels, height, width)
for layer in net:
    x = layer(x)
    print(f"{layer.__class__.__name__:10s} {tuple(x.shape)}")

<details><summary><b>Show the answer</b></summary>

Conv 5×5 with padding 2 and stride 1 keeps the size: (40 + 4 − 5)/1 + 1 = 40 → (4, 12, 40, 40). ReLU does not change shapes.
Max pool → 20. Conv 3×3, stride 2, padding 1: ⌊(20 + 2 − 3)/2⌋ + 1 = 10 → (4, 24, 10, 10). Flatten: 24·10·10 = 2,400 → (4, 2400).
The batch size 4 passes through unchanged.

</details>

### Exercise 1.2 – The formula, checked

**Compute the output size by hand for each line, then check it against PyTorch.**

In [ ]:
def out_size(n, k, s, p):
    return (n + 2 * p - k) // s + 1

for n, k, s, p in [(28, 5, 1, 0), (28, 3, 2, 1), (64, 7, 2, 3), (33, 4, 2, 1)]:
    conv = nn.Conv2d(1, 1, kernel_size=k, stride=s, padding=p)
    real = conv(torch.zeros(1, 1, n, n)).shape[-1]
    print(f"N={n}, K={k}, S={s}, P={p} -> formula {out_size(n, k, s, p)}, PyTorch {real}")

<details><summary><b>Show the answer</b></summary>

24, 14, 32 and 16. In the last line (33 + 2 − 4)/2 = 15.5 is rounded **down** to 15, then + 1 = 16.
“Same” padding for stride 1 is P = (K − 1)/2.

</details>

### Exercise 1.3 – When the input size changes

**The head below was built for 40×40 images. What happens with a 48×48 image? And with the second head?**

In [ ]:
head_flat = nn.Sequential(net, nn.Linear(in_features=2400, out_features=10))
head_gap = nn.Sequential(net[:-1],                         # all layers except Flatten
                         nn.AdaptiveAvgPool2d(output_size=1),  # global average pooling
                         nn.Flatten(),
                         nn.Linear(in_features=24, out_features=10))
for size in (40, 48):
    x = torch.randn(4, 3, size, size)
    try:
        print(size, "flatten head:", head_flat(x).shape)
    except RuntimeError as e:
        print(size, "flatten head: RuntimeError:", str(e)[:90])
    print(size, "GAP head:    ", head_gap(x).shape)

<details><summary><b>Show the answer</b></summary>

For 48×48 the feature map is 24×12×12, so `Flatten` gives 3,456 values, but `Linear` was built for 2,400 inputs:
PyTorch raises a shape-mismatch error. Global average pooling turns *any* 24×H×W map into 24 numbers, so the second head works
for both sizes. (`net[:-1]` means all layers of `net` except the last one.)

</details>

---
## Part 2 – Counting parameters

**Formulas.** $K$ = kernel size, $C_{in}$/$C_{out}$ = input/output channels. Biases are on by default (`bias=True`); with
`bias=False` drop the bias term.

| Layer | Learnable parameters |
|---|---|
| `Linear(D_in, D_out)` | $D_{in} \cdot D_{out} + D_{out}$ |
| `Conv2d(C_in, C_out, K)` | $K \cdot K \cdot C_{in} \cdot C_{out} + C_{out}$ – one $K \times K \times C_{in}$ filter (+ 1 bias) per output channel |
| `Conv2d(..., groups=g)` | $K \cdot K \cdot \frac{C_{in}}{g} \cdot C_{out} + C_{out}$; depthwise ($g = C_{in} = C_{out} = C$): $K \cdot K \cdot C + C$ |
| `ConvTranspose2d(C_in, C_out, K)` | $K \cdot K \cdot C_{in} \cdot C_{out} + C_{out}$ (as for `Conv2d`) |
| `BatchNorm1d/2d(C)` | $2C$ ($\gamma$ and $\beta$); plus $2C$ buffers (running mean and variance) that are **not** parameters |
| `MaxPool2d`, `AvgPool2d`, `Upsample`, `ReLU`, `Dropout`, `Flatten` | $0$ |

- The number of parameters of a convolution does **not** depend on the image size (parameter sharing); that of a `Linear`
  layer after `Flatten` does.
- `count(m)` below adds up the number of elements of all parameter tensors of a module.

### Exercise 2.1 – Linear layers

**What is printed?**

In [ ]:
print(count(nn.Linear(in_features=100, out_features=20)),
      count(nn.Linear(in_features=100, out_features=20, bias=False)))

<details><summary><b>Show the answer</b></summary>

100·20 weights + 20 biases = 2,020; without bias 2,000.

</details>

### Exercise 2.2 – Convolution layers

**What is printed?**

In [ ]:
print(count(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)),
      count(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, bias=False)),
      count(nn.Conv2d(in_channels=16, out_channels=32, kernel_size=1)))

<details><summary><b>Show the answer</b></summary>

Each of the 32 filters has 3·3·16 = 144 weights + 1 bias: 32·145 = 4,640. Without bias 4,608. A 1×1 conv: 32·(16 + 1) = 544.
The number of parameters does **not** depend on the image size (parameter sharing).

</details>

### Exercise 2.3 – BatchNorm: parameters and buffers

**What is printed?**

In [ ]:
bn = nn.BatchNorm2d(num_features=32)
print("parameters:", count(bn))
for name, buf in bn.named_buffers():
    print("buffer:", name, buf.numel())

<details><summary><b>Show the answer</b></summary>

γ and β per channel: 2·32 = 64 learnable parameters. The running mean and variance (32 each) are **buffers**: they are
updated during training, but not by gradient descent, and `parameters()` does not include them. (`num_batches_tracked` is just a
counter.)

</details>

### Exercise 2.4 – Depthwise separable convolution

**What is printed (the last number rounded)?**

In [ ]:
dw  = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1, groups=32, bias=False)  # depthwise
pw  = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=1, bias=False)                        # pointwise
std = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, bias=False)             # standard
print(count(dw) + count(pw), count(std), round(count(std) / (count(dw) + count(pw)), 1))

<details><summary><b>Show the answer</b></summary>

`groups=32` gives each input channel its own 3×3 filter: 32·9 = 288. Pointwise 1×1: 32·64 = 2,048. Together 2,336, versus
3·3·32·64 = 18,432 for a standard conv: about 7.9× fewer.

</details>

### Exercise 2.5 – Learnable vs fixed upsampling

**What are the two printed lines?**

In [ ]:
up1 = nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2, stride=2)
up2 = nn.Upsample(scale_factor=2, mode="nearest")
x = torch.randn(1, 32, 10, 10)
print(up1(x).shape, count(up1))
print(up2(x).shape, count(up2))

<details><summary><b>Show the answer</b></summary>

Both double the size to 20×20. The transpose convolution learns 32·16·2·2 = 2,048 weights + 16 biases = 2,064 and changes the
channels to 16; nearest-neighbour upsampling only copies values: no parameters, 32 channels.

</details>

### Exercise 2.6 – Two classifier heads

**The last feature map is 128×4×4 and there are 10 classes. What is printed?**

In [ ]:
flat_head = nn.Sequential(nn.Flatten(), nn.Linear(in_features=128 * 4 * 4, out_features=10))
gap_head  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_features=128, out_features=10))
print(count(flat_head), count(gap_head))

<details><summary><b>Show the answer</b></summary>

Flatten: 2,048·10 + 10 = 20,490. Global average pooling: 128·10 + 10 = 1,290 – and it works for any input size.

</details>

---
## Part 3 – Beyond CNNs

**Formulas and shapes.**

- **1×1 convolution:** `Conv2d(C_in, C_out, kernel_size=1)` keeps $H \times W$ and maps the $C_{in}$ values at each position
  to $C_{out}$ values – a `Linear` layer applied at every position. Parameters: $C_{in} \cdot C_{out} + C_{out}$.
- **RNN** `nn.RNN(input_size=D, hidden_size=Hd, num_layers=L, batch_first=True)` on input $(N, T, D)$:
  output `out` has shape $(N, T, H_d)$ (last layer, every time step) and `h` has shape $(L, N, H_d)$ (last time step, every layer).
  Update: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$, with the same weights at every time step.
- **ViT tokens:** an $H \times W$ image cut into $p \times p$ patches gives $\frac{H}{p} \cdot \frac{W}{p}$ patch tokens, plus 1 for the
  CLS token. The patch embedding is `Conv2d(3, D, kernel_size=p, stride=p)`.
- **Scaled dot-product attention:** $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$,
  where $d_k$ is the length of a key; the softmax is taken over the keys, so each row of weights sums to 1.

### Exercise 3.1 – A 1×1 convolution as classifier

**The classifier is a 1×1 convolution instead of a `Linear` layer. What are the three printed shapes?**

In [ ]:
backbone = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(kernel_size=2),
)
classifier = nn.Conv2d(in_channels=32, out_channels=5, kernel_size=1)    # 5 classes
print(classifier(backbone(torch.randn(1, 3, 32, 32))).shape)
scores = classifier(backbone(torch.randn(1, 3, 64, 96)))
print(scores.shape)
print(scores.mean(dim=(2, 3)).shape)

<details><summary><b>Show the answer</b></summary>

Two pools divide H and W by 4: 32×32 → 8×8, 64×96 → 16×24. The 1×1 conv gives a score map with 5 channels – one score
vector per position (like a sliding window). Averaging over dims 2 and 3 (height, width) gives one score per class.

</details>

### Exercise 3.2 – RNN outputs

**What is printed?**

In [ ]:
rnn = nn.RNN(input_size=6, hidden_size=12, batch_first=True)
x = torch.randn(4, 9, 6)                 # 4 sequences, 9 time steps, 6 features
out, h = rnn(x)
print(out.shape, h.shape)
rnn2 = nn.RNN(input_size=6, hidden_size=12, num_layers=2, batch_first=True)
print(rnn2(x)[1].shape)

<details><summary><b>Show the answer</b></summary>

`out` holds the hidden state of the last layer at **every** time step: (4, 9, 12). `h` holds the **final** hidden state of
**every layer**: (num_layers, batch, hidden) = (1, 4, 12), and (2, 4, 12) with two layers. The same weights are used at every step.

</details>

### Exercise 3.3 – From image to ViT tokens

**What are the three printed shapes?**

In [ ]:
patch_embed = nn.Conv2d(in_channels=3, out_channels=192, kernel_size=8, stride=8)   # 8×8 patches
x = torch.randn(2, 3, 64, 64)
t = patch_embed(x)
print(t.shape)
tokens = t.flatten(2).transpose(1, 2)            # (batch, number of patches, embedding size)
print(tokens.shape)
cls = torch.zeros(1, 1, 192).expand(2, -1, -1)   # one CLS token per image
print(torch.cat([cls, tokens], dim=1).shape)

<details><summary><b>Show the answer</b></summary>

Kernel = stride = 8 cuts the image into (64/8)² = 64 non-overlapping patches and projects each to 192 numbers: (2, 192, 8, 8).
`flatten(2)` → (2, 192, 64), `transpose(1, 2)` → (2, 64, 192): a sequence of 64 tokens. With the CLS token in front: 65 tokens.

</details>

### Exercise 3.4 – Scaled dot-product attention by hand

**One query, three keys and values. Compute the attention weights by hand first (√2 ≈ 1.414, e^0.707 ≈ 2.03, e^1.414 ≈ 4.11).**

In [ ]:
q = torch.tensor([[0.0, 1.0]])
K = torch.tensor([[0.0, 1.0], [1.0, 0.0], [0.0, 2.0]])
V = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
d_k = K.shape[1]
w = torch.softmax(q @ K.T / math.sqrt(d_k), dim=-1)
print(w)
print(w @ V)

<details><summary><b>Show the answer</b></summary>

Scores q·kᵢ = (1, 0, 2); divided by √2: (0.707, 0, 1.414); softmax: (0.284, 0.140, 0.576) – the third
key matches the query best. The output is the weighted sum of the values: (0.860, 0.716).

</details>